Parameter sweep on the regime multiplier. Grid-search the stressed and crisis multipliers (calm held fixed at 1.0) and plot Sharpe as a heatmap instead of picking a single best combination, then rerun the same sweep on four OOS sub-periods to check whether the current hand-picked (0.25, 0.0) sits in a stable region or was fit to one historical path.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.vrp_strategy.backtest import raw_strategy_returns, vol_target_scale
from src.vrp_strategy.metrics import performance_metrics

DATA_PROCESSED = ROOT / "data" / "processed"
data = pd.read_csv(DATA_PROCESSED / "positions.csv", index_col=0, parse_dates=True)
SPLIT = "2016-01-01"
data = data[data.index >= SPLIT]

TARGET_VOL, VOL_WINDOW, SCALAR_CAP = 0.10, 126, 100.0
CALM_MULT = 1.0  # held fixed, only stressed/crisis are swept

print(f"Out-of-sample period: {data.index[0].date()} → {data.index[-1].date()}")
print(f"Rows: {len(data)}")

Out-of-sample period: 2016-01-04 → 2026-05-22
Rows: 2612


In [2]:
def build_position(regime_label, vrp_rank, stressed_mult, crisis_mult):
    mult = regime_label.map({"calm": CALM_MULT, "stressed": stressed_mult, "crisis": crisis_mult})
    return (mult * vrp_rank).clip(0, 1)

def scaled_return_for_combo(stressed_mult, crisis_mult):
    position = build_position(data["regime_label"], data["vrp_rank"], stressed_mult, crisis_mult)
    raw_ret = raw_strategy_returns(position, data["iv_daily"], data["rv_daily"])
    scaled_ret, _ = vol_target_scale(raw_ret, TARGET_VOL, VOL_WINDOW, SCALAR_CAP)
    return scaled_ret.dropna()

def grid_index(grid, value):
    return int(np.argmin(np.abs(grid - value)))

In [3]:
# Current = 04_position_sizing.ipynb today. Prior = the multiplier before the tuning
# commit in git history (git log: "reduced stress regime multiplier ... increased
# Sharpe from 2.74 to 3.05"). Both are reference points, not part of the search.
CURRENT = (0.25, 0.0)
PRIOR   = (0.50, 0.0)

# 0.25 inserted explicitly so CURRENT lands exactly on the grid.
stressed_grid = np.array([0.0, 0.1, 0.2, 0.25, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0])
crisis_grid   = np.array([0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0])

sharpe_grid = np.full((len(stressed_grid), len(crisis_grid)), np.nan)
returns_cache = {}
for i, sm in enumerate(stressed_grid):
    for j, cm in enumerate(crisis_grid):
        ret = scaled_return_for_combo(sm, cm)
        returns_cache[(sm, cm)] = ret
        sharpe_grid[i, j] = performance_metrics(ret)["sharpe"]

print(f"Swept {sharpe_grid.size} (stressed, crisis) combinations over the full OOS period")

Swept 132 (stressed, crisis) combinations over the full OOS period


In [4]:
best_idx = np.unravel_index(np.nanargmax(sharpe_grid), sharpe_grid.shape)
best_combo = (stressed_grid[best_idx[0]], crisis_grid[best_idx[1]])
best_sharpe = sharpe_grid[best_idx]

current_sharpe = performance_metrics(returns_cache[CURRENT])["sharpe"]
prior_sharpe = performance_metrics(returns_cache[PRIOR])["sharpe"]
current_pct = (sharpe_grid <= current_sharpe).mean()
plateau = int((sharpe_grid >= best_sharpe - 0.2).sum())

print(f"Best on grid:  stressed={best_combo[0]:.2f}, crisis={best_combo[1]:.2f}  Sharpe={best_sharpe:.3f}")
print(f"Current (0.25, 0.0):  Sharpe={current_sharpe:.3f}  ({current_pct*100:.0f}th percentile of the grid)")
print(f"Prior   (0.50, 0.0):  Sharpe={prior_sharpe:.3f}")
print(f"Combos within 0.2 Sharpe of the best: {plateau} / {sharpe_grid.size}")

Best on grid:  stressed=0.20, crisis=0.00  Sharpe=3.130
Current (0.25, 0.0):  Sharpe=3.080  (98th percentile of the grid)
Prior   (0.50, 0.0):  Sharpe=2.850
Combos within 0.2 Sharpe of the best: 6 / 132


Same sweep, split into four roughly-equal OOS sub-periods (~2.6 years each).

In [5]:
# Reuse the cached full-period return series per combo instead of recomputing the
# vol-target rolling window from scratch per sub-period, so each sub-period's scalar
# still has a proper continuous lead-in rather than its own fresh warmup gap.
ref_index = returns_cache[CURRENT].index
chunks = np.array_split(ref_index, 4)
sub_periods = [(c[0], c[-1]) for c in chunks]

for i, (start, end) in enumerate(sub_periods):
    print(f"Sub-period {i+1}: {start.date()} → {end.date()}  ({len(chunks[i])} rows)")

Sub-period 1: 2016-07-06 → 2018-12-21  (622 rows)
Sub-period 2: 2018-12-24 → 2021-06-11  (621 rows)
Sub-period 3: 2021-06-14 → 2023-11-29  (621 rows)
Sub-period 4: 2023-11-30 → 2026-05-22  (621 rows)


In [6]:
sharpe_grids_sub = []
for start, end in sub_periods:
    grid = np.full((len(stressed_grid), len(crisis_grid)), np.nan)
    for i, sm in enumerate(stressed_grid):
        for j, cm in enumerate(crisis_grid):
            ret_slice = returns_cache[(sm, cm)].loc[start:end]
            grid[i, j] = performance_metrics(ret_slice)["sharpe"]
    sharpe_grids_sub.append(grid)

ci, cj = grid_index(stressed_grid, CURRENT[0]), grid_index(crisis_grid, CURRENT[1])

print(f"{'Sub-period':<24}{'Best combo':<18}{'Best Sharpe':<14}{'Current Sharpe':<17}{'Current pct'}")
for k, (start, end) in enumerate(sub_periods):
    grid = sharpe_grids_sub[k]
    bidx = np.unravel_index(np.nanargmax(grid), grid.shape)
    bcombo = (round(float(stressed_grid[bidx[0]]), 2), round(float(crisis_grid[bidx[1]]), 2))
    bsharpe = grid[bidx]
    csharpe = grid[ci, cj]
    cpct = (grid <= csharpe).mean()
    label = f"{start.date()} to {end.date()}"
    print(f"{label:<24}{str(bcombo):<18}{bsharpe:<14.3f}{csharpe:<17.3f}{cpct*100:.0f}th pct")

print("\nIf the best-combo column jumps around a lot between rows, or the current")
print("combo's percentile swings widely, the full-period optimum is more likely fit")
print("to one historical path than a stable, broad effect.")

Sub-period              Best combo        Best Sharpe   Current Sharpe   Current pct
2016-07-06 to 2018-12-21(0.0, 0.0)        1.352         1.344            75th pct
2018-12-24 to 2021-06-11(0.2, 0.0)        2.090         2.058            99th pct
2021-06-14 to 2023-11-29(0.2, 0.1)        6.379         6.298            98th pct
2023-11-30 to 2026-05-22(0.0, 0.0)        5.466         4.763            98th pct

If the best-combo column jumps around a lot between rows, or the current
combo's percentile swings widely, the full-period optimum is more likely fit
to one historical path than a stable, broad effect.


In [7]:
all_vals = np.concatenate([sharpe_grid.ravel()] + [g.ravel() for g in sharpe_grids_sub])
all_vals = all_vals[~np.isnan(all_vals)]
vmin, vmax = np.percentile(all_vals, [2, 98])

fig, ax = plt.subplots(figsize=(8, 6))
mesh = ax.pcolormesh(crisis_grid, stressed_grid, sharpe_grid, shading="nearest",
                      cmap="viridis", vmin=vmin, vmax=vmax)
ax.set_xlabel("Crisis multiplier")
ax.set_ylabel("Stressed multiplier")
ax.set_title(f"Sharpe by regime multiplier — full OOS period\n"
             f"({data.index[0].date()} to {data.index[-1].date()})")
fig.colorbar(mesh, ax=ax, label="Sharpe")
ax.scatter(CURRENT[1], CURRENT[0], color="red", edgecolor="white", s=140, marker="*", label=f"Current {CURRENT}")
ax.scatter(PRIOR[1], PRIOR[0], color="orange", edgecolor="white", s=100, marker="o", label=f"Prior {PRIOR}")
ax.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.savefig(DATA_PROCESSED / "regime_multiplier_sweep_full.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
for k, ax in enumerate(axes.flat):
    grid = sharpe_grids_sub[k]
    start, end = sub_periods[k]
    mesh = ax.pcolormesh(crisis_grid, stressed_grid, grid, shading="nearest",
                          cmap="viridis", vmin=vmin, vmax=vmax)
    ax.set_title(f"{start.date()} to {end.date()}", fontsize=10)
    ax.set_xlabel("Crisis multiplier")
    ax.set_ylabel("Stressed multiplier")
    ax.scatter(CURRENT[1], CURRENT[0], color="red", edgecolor="white", s=100, marker="*")
fig.suptitle("Sharpe by regime multiplier — four OOS sub-periods (shared color scale)",
             fontsize=13, fontweight="bold")
fig.colorbar(mesh, ax=axes, label="Sharpe", shrink=0.8)
plt.savefig(DATA_PROCESSED / "regime_multiplier_sweep_subperiods.png", dpi=150, bbox_inches="tight")
plt.show()